# Protein Interface GCN - paper-protocol replica (Fout et al., NeurIPS 2017, Table 2)
No-Convolution / Node Average / Node+Edge Average, 10 seeds, median per-complex ROC AUC on the 55 official test complexes. Data: authors' Zenodo record 1127774 (MD5 verified).

In [ ]:
import os,hashlib,urllib.request
os.makedirs('data',exist_ok=True)
F={'train.cpkl.gz':'c9d461dec1c4a71a2a788051fde5b85b','test.cpkl.gz':'801bb8582145af116a0092d8d18a3759'}
for n,h in F.items():
    p='data/'+n
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://zenodo.org/records/1127774/files/{n}?download=1',p)
    assert hashlib.md5(open(p,'rb').read()).hexdigest()==h,n
    print(n,'verified',os.path.getsize(p))

In [ ]:
%%writefile dataset.py
"""Load the verified Python 2 DB5 pickles published with Fout et al. 2017."""
from __future__ import annotations

import gzip
import pickle
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent


class RestrictedNumpyUnpickler(pickle.Unpickler):
    """Accept only the NumPy constructors required by the paper's data files."""

    ALLOWED = {
        ("numpy.core.multiarray", "_reconstruct"),
        ("numpy", "ndarray"),
        ("numpy", "dtype"),
        ("_codecs", "encode"),
        ("copy_reg", "_reconstructor"),
        ("__builtin__", "object"),
    }

    def find_class(self, module, name):
        if (module, name) not in self.ALLOWED:
            raise pickle.UnpicklingError(f"Unexpected class in research dataset: {module}.{name}")
        return super().find_class(module, name)


def load_split(name: str):
    if name not in ("train", "test"):
        raise ValueError(name)
    with gzip.open(ROOT / "data" / f"{name}.cpkl.gz", "rb") as stream:
        codes, cases = RestrictedNumpyUnpickler(stream, encoding="latin1").load()
    if len(codes) != len(cases):
        raise ValueError("PDB code / complex count mismatch")
    for code, case in zip(codes, cases):
        if code != case["complex_code"]:
            raise ValueError(f"Complex code mismatch: {code}")
        for key in ("l_vertex", "r_vertex", "l_hood_indices", "r_hood_indices", "label"):
            if not isinstance(case[key], np.ndarray):
                raise TypeError(f"{code}: {key} is not a NumPy array")
    return codes, cases


if __name__ == "__main__":
    for split in ("train", "test"):
        codes, cases = load_split(split)
        print(split, len(codes), codes[:5], flush=True)
        first = cases[0]
        print({key: (value.shape, str(value.dtype)) for key, value in first.items()
               if isinstance(value, np.ndarray)}, flush=True)


In [ ]:
%%writefile evaluation.py
"""Explicit metric definitions for rare residue-pair contacts.

Ranking uses logits; sigmoid values are diagnostic scores, not calibrated
probabilities. Top-K metrics average uniformly within a tie at the cutoff.
"""
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score, auc, precision_recall_curve, roc_auc_score


def checked_arrays(y, score):
    y, score = np.asarray(y).ravel(), np.asarray(score, dtype=np.float64).ravel()
    if not len(y) or y.shape != score.shape:
        raise ValueError("Labels and scores must have matching nonempty shapes")
    if not np.isin(y, [0, 1]).all() or not np.isfinite(score).all():
        raise ValueError("Expected binary labels and finite scores")
    if not (0 < y.sum() < len(y)):
        raise ValueError("Ranking evaluation requires both label classes")
    return y.astype(np.uint8), score


def independent_auc(y, score):
    """Mann-Whitney/rank-sum definition; tied pairs receive half credit."""
    y, score = checked_arrays(y, score)
    positive, negative = int(y.sum()), int(len(y)-y.sum())
    ranks = rankdata(score, method="average")
    return float((ranks[y == 1].sum() - positive*(positive+1)/2) / (positive*negative))


def independent_ap(y, score):
    """Sum precision * recall increment at distinct score thresholds."""
    y, score = checked_arrays(y, score)
    order = np.argsort(-score, kind="stable")
    sorted_y, sorted_score = y[order], score[order]
    end = np.r_[np.flatnonzero(np.diff(sorted_score) != 0), len(y)-1]
    tp = np.cumsum(sorted_y, dtype=np.int64)[end]
    increment = np.diff(np.r_[0, tp]) / int(y.sum())
    precision = tp / (end+1)
    return float(np.dot(increment, precision))


def topk_expected(y, score, k):
    y, score = checked_arrays(y, score)
    k = min(max(1, int(k)), len(y))
    cutoff = np.partition(score, len(score)-k)[len(score)-k]
    above, tied = score > cutoff, score == cutoff
    slots = k-int(above.sum())
    expected_tp = float(y[above].sum()) + slots*float(y[tied].mean())
    precision = expected_tp/k
    return {"k": k, "expected_tp": expected_tp, "precision": precision,
            "recall": expected_tp/int(y.sum()), "enrichment": precision/float(y.mean()),
            "cutoff_tie_size": int(tied.sum()), "cutoff_tie_slots": slots}


def threshold_metrics(y, score, threshold=0.5):
    y, score = checked_arrays(y, score)
    prediction = score >= threshold
    tp = int(np.sum(prediction & (y == 1)))
    fp = int(np.sum(prediction & (y == 0)))
    fn = int(np.sum(~prediction & (y == 1)))
    tn = int(np.sum(~prediction & (y == 0)))
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall, specificity = tp/(tp+fn), tn/(tn+fp)
    denominator = float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))**0.5
    return {"threshold": threshold, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "predicted_positive": tp+fp, "precision": precision, "recall": recall,
            "specificity": specificity, "false_positive_rate": 1-specificity,
            "f1": 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0,
            "mcc": (tp*tn-fp*fn)/denominator if denominator else 0.0,
            "balanced_accuracy": (recall+specificity)/2,
            "accuracy": (tp+tn)/len(y), "precision_defined": bool(tp+fp)}


def ranking_metrics(y, logits):
    y, logits = checked_arrays(y, logits)
    precision, recall, _ = precision_recall_curve(y, logits)
    return {"roc_auc": float(roc_auc_score(y, logits)),
            "average_precision": float(average_precision_score(y, logits)),
            "pr_auc_trapezoid": float(auc(recall, precision)),
            "standardized_partial_roc_auc_fpr_0_01": float(roc_auc_score(y, logits, max_fpr=0.01)),
            "n_pairs": len(y), "n_contacts": int(y.sum()), "prevalence": float(y.mean())}


def uncalibrated_score_diagnostics(y, logits, scores):
    y, logits = checked_arrays(y, logits)
    scores = np.asarray(scores, dtype=np.float64).ravel()
    if scores.shape != y.shape or not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):
        raise ValueError("Expected one finite sigmoid score in [0,1] per label")
    return {"brier_uncalibrated_score": float(np.mean((scores-y)**2)),
            "log_loss_uncalibrated_score": float(np.mean(np.logaddexp(0, logits)-y*logits)),
            "mean_uncalibrated_score": float(scores.mean())}


In [ ]:
%%writefile paper_replica.py
"""Paper-protocol replica of Fout et al. (NeurIPS 2017), Table 2.

Variants: No-Convolution, Node Average (Eq. 1), Node and Edge Average (Eq. 2).
Paper settings reproduced: 2 conv layers with (256, 512) filters, 512-unit dense
layer after merging, dropout 0.5, 10 random seeds, median per-complex ROC AUC on
the official 55-complex test set, 175 development complexes for final training.
Deviation (disclosed): Adam instead of SGD, one complex per step, epoch count
chosen on a 140/35 development split with seed 224 and never on test data.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from torch import nn

from dataset import load_split
from evaluation import topk_expected

OUT = Path(__file__).resolve().parent / "results"
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VARIANTS = ("noconv", "node", "nodeedge")


class Case:
    def __init__(self, raw):
        t = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt, device=DEV)
        self.code = str(raw["complex_code"])
        self.xl, self.xr = t(raw["l_vertex"]), t(raw["r_vertex"])
        self.hl = t(raw["l_hood_indices"], torch.long).squeeze(-1)
        self.hr = t(raw["r_hood_indices"], torch.long).squeeze(-1)
        self.el, self.er = t(raw["l_edge"]), t(raw["r_edge"])
        lab = np.asarray(raw["label"], dtype=np.int64)
        assert lab.shape[1] == 3 and np.isin(lab[:, 2], [-1, 1]).all()
        self.pair = torch.as_tensor(lab[:, :2], device=DEV)
        self.y = (lab[:, 2] == 1).astype(np.uint8)
        self.yt = torch.as_tensor(self.y, dtype=torch.float32, device=DEV)


def load(split):
    return [Case(r) for r in load_split(split)[1]]


class Layer(nn.Module):
    def __init__(self, d_in, d_out, variant):
        super().__init__()
        self.variant = variant
        self.center = nn.Linear(d_in, d_out)
        if variant != "noconv":
            self.neigh = nn.Linear(d_in, d_out, bias=False)
        if variant == "nodeedge":
            self.edge = nn.Linear(2, d_out, bias=False)

    def forward(self, x, hood, edge):
        z = self.center(x)
        if self.variant != "noconv":
            z = z + self.neigh(x[hood].mean(1))
        if self.variant == "nodeedge":
            z = z + self.edge(edge.mean(1))
        return torch.relu(z)


class Net(nn.Module):
    def __init__(self, variant, filters=(256, 512), dense=512, p=0.5):
        super().__init__()
        dims = (70,) + tuple(filters)
        self.layers = nn.ModuleList(Layer(a, b, variant) for a, b in zip(dims, dims[1:]))
        self.drop = nn.Dropout(p)
        self.head = nn.Sequential(nn.Linear(2 * dims[-1], dense), nn.ReLU(),
                                  nn.Dropout(p), nn.Linear(dense, 1))

    def enc(self, x, hood, edge):
        for layer in self.layers:
            x = self.drop(layer(x, hood, edge))
        return x

    def embed(self, c):
        return self.enc(c.xl, c.hl, c.el), self.enc(c.xr, c.hr, c.er)

    def score(self, a, b, pair):
        return self.head(torch.cat((a[pair[:, 0]], b[pair[:, 1]]), 1)).squeeze(-1)


def train_epoch(model, cases, opt, rng):
    model.train()
    for i in rng.permutation(len(cases)):
        c = cases[int(i)]
        opt.zero_grad(set_to_none=True)
        a, b = model.embed(c)
        loss = nn.functional.binary_cross_entropy_with_logits(
            model.score(a, b, c.pair), c.yt, pos_weight=torch.tensor(10.0, device=DEV))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()


@torch.no_grad()
def predict(model, c, chunk=262144):
    model.eval()
    a, b = model.embed(c)
    out = [torch.sigmoid(model.score(a, b, c.pair[s:s + chunk])) for s in range(0, len(c.pair), chunk)]
    return torch.cat(out).cpu().numpy()


def evaluate(model, cases):
    per, ys, ss = {}, [], []
    for c in cases:
        s = predict(model, c)
        per[c.code] = {"auc": float(roc_auc_score(c.y, s)),
                       "ap": float(average_precision_score(c.y, s)),
                       "p10": float(topk_expected(c.y, s, 10)["precision"])}
        ys.append(c.y); ss.append(s)
    y, s = np.concatenate(ys), np.concatenate(ss)
    v = lambda k: [d[k] for d in per.values()]
    return {"median_auc": float(np.median(v("auc"))), "mean_auc": float(np.mean(v("auc"))),
            "median_ap": float(np.median(v("ap"))), "mean_p10": float(np.mean(v("p10"))),
            "pooled_ap": float(average_precision_score(y, s)),
            "pooled_auc": float(roc_auc_score(y, s)), "per_complex": per}


def seed_all(seed):
    np.random.seed(seed); torch.manual_seed(seed)


def select_epoch(variant, dev, max_epochs, lr):
    perm = np.random.default_rng(224).permutation(len(dev))
    train = [dev[int(i)] for i in perm[:140]]
    valid = [dev[int(i)] for i in perm[140:]]
    seed_all(224)
    model = Net(variant).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    rng = np.random.default_rng(224)
    hist = []
    for ep in range(1, max_epochs + 1):
        t0 = time.time()
        train_epoch(model, train, opt, rng)
        auc = float(np.median([roc_auc_score(c.y, predict(model, c)) for c in valid]))
        hist.append(auc)
        print(f"[{variant}] select epoch {ep}: val median AUC {auc:.4f} ({time.time()-t0:.1f}s)", flush=True)
    best = int(np.argmax(hist)) + 1
    return best, hist


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variants", nargs="+", default=list(VARIANTS))
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--max-epochs", type=int, default=40)
    ap.add_argument("--lr", type=float, default=5e-4)
    args = ap.parse_args()
    OUT.mkdir(exist_ok=True)
    dev, test = load("train"), load("test")
    assert not {c.code for c in dev} & {c.code for c in test}
    print(f"dev {len(dev)} test {len(test)} device {DEV}", flush=True)
    for variant in args.variants:
        path = OUT / f"replica_{variant}.json"
        rep = json.loads(path.read_text()) if path.exists() else {}
        if "best_epoch" not in rep:
            rep["best_epoch"], rep["selection_val_median_auc"] = select_epoch(
                variant, dev, args.max_epochs, args.lr)
            rep["runs"] = []
            path.write_text(json.dumps(rep))
        print(f"[{variant}] selected epoch {rep['best_epoch']}", flush=True)
        for seed in range(len(rep["runs"]), args.seeds):
            t0 = time.time()
            seed_all(1000 + seed)
            model = Net(variant).to(DEV)
            opt = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-4)
            rng = np.random.default_rng(1000 + seed)
            for _ in range(rep["best_epoch"]):
                train_epoch(model, dev, opt, rng)
            res = evaluate(model, test)
            rep["runs"].append({"seed": 1000 + seed, **res})
            path.write_text(json.dumps(rep))
            print(f"[{variant}] seed {seed}: median AUC {res['median_auc']:.4f} "
                  f"AP {res['median_ap']:.4f} ({time.time()-t0:.0f}s)", flush=True)
        a = [r["median_auc"] for r in rep["runs"]]
        print(f"[{variant}] {len(a)} runs: median AUC {np.mean(a):.4f} +/- {np.std(a, ddof=1):.4f}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import os
os.makedirs('results',exist_ok=True)
!python -u paper_replica.py

In [ ]:
import json,glob,numpy as np
for f in sorted(glob.glob('results/replica_*.json')):
    r=json.load(open(f));a=[x['median_auc'] for x in r['runs']]
    print(f, 'epoch',r['best_epoch'],'n',len(a),'median AUC %.4f +/- %.4f'%(np.mean(a),np.std(a,ddof=1)),'AP %.4f'%np.mean([x['median_ap'] for x in r['runs']]))
    print(json.dumps({k:v for k,v in r.items() if k!='runs'})[:0])